# Prophet 单序列原型

**目的**：对代表性序列跑通 Prophet 建模流程，验证节假日特征接入、extra regressor、评估接口。

**序列**：
1. (1, GROCERY I) — 主力高销量
2. (44, BEVERAGES) — 稀疏序列但有促销

In [ ]:
import sys, os, warnings
sys.path.append(os.path.abspath('..'))
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.models.utils import load_time_series, time_split, evaluate_all, build_holiday_df, load_oil
from src.models.baseline import SeasonalNaive
from src.models.prophet_model import ProphetModel

%matplotlib inline
plt.style.use('ggplot')

In [ ]:
s1 = load_time_series(1, 'GROCERY I')
train, val, test = time_split(s1)
oil = load_oil()
holidays_df = build_holiday_df()

fig, ax = plt.subplots(figsize=(14, 4))
train['sales'].plot(ax=ax, label='Train')
val['sales'].plot(ax=ax, label='Validation')
test['sales'].plot(ax=ax, label='Test')
ax.set_title('Store 1, GROCERY I — Sales')
ax.legend()
plt.tight_layout()
plt.savefig('../output/prophet_s1_timeseries.png', dpi=100)
plt.show()

print(f'Train: {len(train)} days, Val: {len(val)} days, Test: {len(test)} days')
print(f'Holidays: {len(holidays_df)} rows')

In [ ]:
# Baseline for comparison
baseline = SeasonalNaive(period=7)
baseline.fit(train['sales'])
preds_seas = baseline.predict(len(val))
metrics_seas = evaluate_all(val['sales'].values, preds_seas)
print('Seasonal Naive Val:', metrics_seas)

In [ ]:
model = ProphetModel(
    holidays_df=holidays_df,
    extra_regressors=['onpromotion', 'dcoilwtico'],
)

train_df = model.prepare_data(train, oil_series=oil)
model.fit(train_df)
print('Model trained successfully')

In [ ]:
# 准备未来数据
future = val.reset_index()[['date', 'onpromotion']].rename(columns={'date': 'ds'})
oil_df = oil.reset_index()
oil_df.columns = ['ds', 'dcoilwtico']
future = future.merge(oil_df, on='ds', how='left')

forecast = model.predict(periods=0, future_df=future)
y_pred = model.get_yhat(forecast)
metrics_val = evaluate_all(val['sales'].values, y_pred)
print('Prophet Val:', metrics_val)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))
val['sales'].plot(ax=ax, label='Actual', linewidth=2)
pd.Series(y_pred, index=val.index).plot(ax=ax, label='Prophet Pred', linestyle='--')
pd.Series(preds_seas, index=val.index).plot(ax=ax, label='Seasonal Naive', linestyle=':')
ax.set_title('Prophet — Validation Predictions vs Actual')
ax.legend()
plt.tight_layout()
plt.savefig('../output/prophet_s1_prediction.png', dpi=100)
plt.show()

In [ ]:
# 测试集预测
future_test = test.reset_index()[['date', 'onpromotion']].rename(columns={'date': 'ds'})
future_test = future_test.merge(oil_df, on='ds', how='left')
forecast_test = model.predict(periods=0, future_df=future_test)
y_pred_test = model.get_yhat(forecast_test)
metrics_test = evaluate_all(test['sales'].values, y_pred_test)
print('Prophet Test:', metrics_test)

In [ ]:
s3 = load_time_series(44, 'BEVERAGES')
t3, v3, te3 = time_split(s3)

model3 = ProphetModel(
    holidays_df=holidays_df,
    extra_regressors=['onpromotion', 'dcoilwtico'],
)
train3_df = model3.prepare_data(t3, oil_series=oil)
model3.fit(train3_df)

future3 = v3.reset_index()[['date', 'onpromotion']].rename(columns={'date': 'ds'})
future3 = future3.merge(oil_df, on='ds', how='left')
forecast3 = model3.predict(periods=0, future_df=future3)
y_pred3 = model3.get_yhat(forecast3)
m3_val = evaluate_all(v3['sales'].values, y_pred3)
print(f'S3 (44, BEVERAGES) Prophet Val: {m3_val}')